# [실습] 80% 예측구간 포함률 — 날씨 외생 변수 RMSE

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## Prophet 을 운영 예측에 쓸 수 있나

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">시계열 예측 · 다섯째 날</div>
<h1 style="color:#000000;margin:10px 0">1년 추세의 변화점과 날씨 외생 변수</h1>
<div style="font-size:1.05em;color:#656565">Prophet 기본 구성을 예측 원점 5개에서 베이스라인과 비교하고, 80% 예측구간의 실제 포함률과 기온·강수 외생 변수의 효과를 직접 확인합니다</div>
</div>

**오늘의 질문** Prophet 을 운영 예측에 써도 될까요? 쓴다면 어떤 구성이어야 할까요?
**예측 원점(forecast origin)** 5개의 평균 RMSE 로 답합니다. 비교 대상은 **베이스라인(baseline)** 인 계절 나이브 168 로, 1주일 전 같은 시각 값을 그대로 씁니다. 이어서 Prophet 의 80% 예측구간과 기온·강수 외생 변수를 보고서에 어떻게 적을지 직접 확인합니다.

1. 맨 위 설치 셀을 한 번 실행합니다. 런타임 유형은 CPU 로 둡니다. `prophet` 설치에 1～2분 걸리고, 설치 뒤 `cmdstanpy` 오류가 나면 런타임을 다시 시작해 설치 셀부터 다시 실행합니다.
2. 「오늘의 준비」 셀을 실행해 두고, 출력을 기다리는 동안 개념 확인 두 문항을 풉니다.
3. 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다.

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.7" "prophet==1.3.0" "statsmodels==0.14.6"

import os, json, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from prophet import Prophet
import warnings; warnings.filterwarnings("ignore")
import cmdstanpy; cmdstanpy.utils.get_logger().disabled = True   # 적합 때마다 찍히는 진행 로그를 끈다
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 베이스라인=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다
SEED = 42        # Prophet 의 구간은 시뮬레이션이라 시드를 고정해야 본문 숫자가 재현된다


def datefmt(ax, fmt="%m-%d", n=5):
    """시간 축 눈금 개수와 표기를 잡아 날짜 라벨이 겹치지 않게 한다"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=n))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def rmse(a, b):
    """두 계열의 평균 제곱근 오차 — ML 수업에서 쓰던 그 지표 그대로"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))))

print("사용 폰트:", FONT, "| 색 팔레트 준비 완료 — 채점 셀은 LMS 가 자동으로 준비합니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 코드 행들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판단에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6eyJ2YWx1ZXMiOnsi66qF66qpIO2PrO2VqOuloCI6Wzc5LjUsODAuNSwiJSJdLCLsi6TsoJwg7Y+s7ZWo66WgIjpbODUuMCw5NS4w"
           "LCIlIl19LCJwbG90Ijp7ImtpbmQiOiJsaW5lIiwibl9zZXJpZXMiOlsyLDRdLCJtaW5fcG9pbnRzIjoxMDAsIm5fcG9pbnRzIjpbMTY4LDIw"
           "MF0sIuy2leudvOuyqCI6dHJ1ZX19LCJtMiI6eyJ2YWx1ZXMiOnsi64Kg7JSo66W8IOuNlO2VnCDqsoDspp0gMTY47Iuc6rCEIFJNU0UiOlsz"
           "MzQuMCwzNDguMCwi64yAIl0sIuq4sOyYqCDqs4TsiJgiOlszOS4wLDQxLjUsIuuMgC/rj4QiXX0sInBsb3QiOnsia2luZCI6ImJhcmgiLCJu"
           "IjozLCLrp4nrjIDqsJIiOltbMTk1LDIxNV0sWzMwMCwzMjJdLFszMzQsMzUwXV0sIuqwgOuhnOy2leudvOuyqCI6dHJ1ZX19fQ==")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuT"
           "pOyWtOyguCDsnojripQgdHJhaW4sIHZhbGlkLCBob2xpZGF5c19vbmUsIFNFRUQsIHJtc2Ug66W8IOq3uOuMgOuhnCDsk7Dqs6Ag7IOI66Gc"
           "IOunjOuTpOyngOuKlCDrp5DslYQg7KSYLiB0cmFpbiDsnZgg7J24642x7Iqk66W8IGRzLAogICAgZGVtYW5kIOyXtOydhCB5IOuhnCDsmK7q"
           "uLQg7ZGc66GcIFByb3BoZXQg7J2EIOygge2Vqe2VtCDspJguIOyXsOqwhCDqs4TsoIjshLHsnYAg7KCc7Jm47ZWY6rOgIOyjvOqwhMK37J28"
           "6rCEIOqzhOygiOyEseydgCDtj6ztlajtlZjqs6AgaG9saWRheXNfb25lIOydhCDtnLTsnbzroZwg65GQ6rOgLAogICAg7KCB7ZWpIOyngeyg"
           "hOqzvCDsmIjsuKEg7KeB7KCE7JeQIG5wLnJhbmRvbS5zZWVkKFNFRUQpIOulvCDrtojrn6wg7KSYLiB2YWxpZCAxNjjsi5zqsIHsnYQg7JiI"
           "7Lih7ZWcIOuSpCAi66qF66qpIO2PrO2VqOuloCA9IOqwkiUiIOuKlCDsoIHtlantlZwg66qo7ZiV7J2YCiAgICBpbnRlcnZhbF93aWR0aCDr"
           "pbwg7J297Ja0IOygleyImOuhnCwgIuyLpOygnCDtj6ztlajrpaAgPSDqsJIlICjrqocvMTY47Iuc6rCEKSIg7J2AIOyLpOy4oeydtCB5aGF0"
           "X2xvd2VyIOydtOyDgSB5aGF0X3VwcGVyIOydtO2VmOyduCDruYTsnKjroZwg7IaM7IiYCiAgICDrkZjsp7gg7J6Q66as6rmM7KeAIO2VnCDt"
           "lonslKkgcHJpbnQg7ZW0IOykmC4g6rWs6rCEIOyVhOuemOuhnCDrspfslrTrgpwg7Iuc6rCBIOyImOyZgCDsnITroZwg67KX7Ja064KcIOyL"
           "nOqwgSDsiJjrj4Qg7ZWcIO2WieyXkCDssI3slrQg7KSYLiDqt7jrnpjtlITripQg7ZWcIOyepeunjCwKICAgIOqygOymnSAxNjjsi5zqsITs"
           "nZgg7Iuk7Lih7ISg6rO8IOyYiOy4oeyEoOydhCDqt7jrpqzqs6Ag6rWs6rCE7J2EIOyYheqyjCDsuaDtlZwg65KkIOqwgOuhnOy2lcK37IS4"
           "66Gc7LaVIOydtOumhOqzvCDrspTroYDrpbwg64us7JWEIOykmC4KCiAgICB0cl90YmwgPSBwZC5EYXRhRnJhbWUoeyJkcyI6IHRyYWluLmlu"
           "ZGV4LCAieSI6IHRyYWluWyJkZW1hbmQiXS52YWx1ZXN9KQogICAgbnAucmFuZG9tLnNlZWQoU0VFRCkgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgICAgICAgICAgICMg6rWs6rCE7J2AIO2RnOuzuCDqsr3roZzroZwg66eM65Og64ukCiAgICBtID0gUHJvcGhldCh5ZWFybHlfc2Vh"
           "c29uYWxpdHk9RmFsc2UsIHdlZWtseV9zZWFzb25hbGl0eT1UcnVlLAogICAgICAgICAgICAgICAgZGFpbHlfc2Vhc29uYWxpdHk9VHJ1ZSwg"
           "aG9saWRheXM9aG9saWRheXNfb25lKQogICAgbS5maXQodHJfdGJsKQogICAgbnAucmFuZG9tLnNlZWQoU0VFRCkKICAgIHAgPSBtLnByZWRp"
           "Y3QocGQuRGF0YUZyYW1lKHsiZHMiOiB2YWxpZC5pbmRleH0pKQoKICAgIGFjdCA9IHZhbGlkWyJkZW1hbmQiXS52YWx1ZXMKICAgIGxvLCBo"
           "aSA9IHBbInloYXRfbG93ZXIiXS52YWx1ZXMsIHBbInloYXRfdXBwZXIiXS52YWx1ZXMKICAgIGluc2lkZSA9IChhY3QgPj0gbG8pICYgKGFj"
           "dCA8PSBoaSkgICAgICAgICAgICAgICAgICAgICAgICAjIO2VmO2VnCDsnbTsg4HsnbTqs6Ag7IOB7ZWcIOydtO2VmAogICAgcHJpbnQoZiLr"
           "qoXrqqkg7Y+s7ZWo66WgID0ge20uaW50ZXJ2YWxfd2lkdGg6LjAlfSIpICAgICAgICAgICAgICMg7Iir7J6Q66GcIOyggeyngCDslYrqs6Ag"
           "66qo7ZiV7JeQ7IScIOydveuKlOuLpAogICAgcHJpbnQoZiLsi6TsoJwg7Y+s7ZWo66WgID0ge2luc2lkZS5tZWFuKCk6LjIlfSAgICh7aW5z"
           "aWRlLnN1bSgpfS97bGVuKGFjdCl97Iuc6rCEKSIpCiAgICBwcmludChmIuq1rOqwhCDslYTrnpjroZwg67KX7Ja064KcIOyLnOqwgSB7KGFj"
           "dCA8IGxvKS5zdW0oKX3si5zqsIQgwrcg7JyE66GcIOuyl+yWtOuCnCDsi5zqsIEgeyhhY3QgPiBoaSkuc3VtKCl97Iuc6rCEIikKCiAgICBm"
           "aWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XLCAzLjQpKQogICAgYXguZmlsbF9iZXR3ZWVuKHZhbGlkLmluZGV4LCBsbywg"
           "aGksIGNvbG9yPUNPTFsi7ZWY64qY7IOJIl0sIGFscGhhPTAuMjIsIGxhYmVsPSI4MCUg7JiI7Lih6rWs6rCEIikKICAgIGF4LnBsb3QodmFs"
           "aWQuaW5kZXgsIGFjdCwgY29sb3I9Q09MWyLrgqjsg4kiXSwgbHc9MS4wLCBsYWJlbD0i7Iuk7LihIikKICAgIGF4LnBsb3QodmFsaWQuaW5k"
           "ZXgsIHBbInloYXQiXSwgY29sb3I9Q09MWyLtlZjripjsg4kiXSwgbHc9MS4yLCBsYWJlbD0i7JiI7LihIikKICAgIGF4LnNldF94bGFiZWwo"
           "IuqygOymnSAxNjjsi5zqsIQiKQogICAgYXguc2V0X3lsYWJlbCgi7IiY7JqUKOuMgCkiKQogICAgZGF0ZWZtdChheCwgIiVtLSVkIiwgNykK"
           "ICAgIGF4LmxlZ2VuZChsb2M9InVwcGVyIHJpZ2h0IikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDsmIjsg4Eg7IWLIOqwgOyatOuNsCDr"
           "i7XsnYAg65GQIOuyiOynuOyeheuLiOuLpC4g66qF66qpIO2PrO2VqOuloCA4MCUg7JeQIOyLpOygnCDtj6ztlajrpaDsnYAg7JW9IDkwJSgx"
           "Njjsi5zqsIQg6rCA7Jq0642wIOyVvSAxNTLsi5zqsIQp6528LCDsnbQg7KO87JeQ64qUIOq1rOqwhOydtCDtlYTsmpTrs7Tri6Qg64ST7JeI"
           "7Iq164uI64ukLiDrspfslrTrgpwg7JW9IDE27Iuc6rCE7J2AIOyVhOuemOuhnCDslb0gMTHsi5zqsIQsIOychOuhnCDslb0gNeyLnOqwhOyd"
           "tOudvCDtlZzsqr3snLzroZwg66qw66as7KeAIOyViuyVmOyKteuLiOuLpC4KLSDrhJPqsowg64KY7JioIOq5jOuLreydgCDqtazqsIQg6ri4"
           "7J2066W8IO2VmeyKtSAxLDg0OOyLnOqwhOydmCDsnpTssKjqsIAg7KCV7ZWY6riwIOuVjOusuOyeheuLiOuLpC4g7ZWZ7Iq1IOyelOywqCDt"
           "kZzspIDtjrjssKjqsIAg7JW9IDQwNOuMgOudvCA4MCUg6rWs6rCE7J2AIOyYiOy4oeqwkiDsnITslYTrnpjroZwg7JW9IDUxN+uMgCgxLjI4"
           "67CwKeydtOqzoCwg7J20IOyjvOydmCBSTVNFIOyVvSAzMTDrjIDrs7Tri6Qg64ST7Iq164uI64ukLiDsnbTroaDsl5DshJwg67O4IDEx7JuU"
           "IDE37J28KDMzNuyLnOqwhCDsoIHtlakp7J2AIOyYiOy4oeyEoOydtCDsi6TsuKHrs7Tri6Qg64aS7JWEIDUwJSDsmIDsirXri4jri6QuIDEs"
           "ODQ47Iuc6rCEIOygge2VqeydgCDtlZwg7KO8IOyYiOy4oSDtj4nqt6DsnbQg7JW9IDcwN+uMgOuhnCDsi6TsuKEg7Y+J6regIDY5NC4yNOuM"
           "gOyZgCDqsIDquZ3sirXri4jri6QuCi0g7Z2U7ZWcIOyLpOyImCDtlZjrgpjripQg66qF66qpIO2PrO2VqOuloOydhCA5NSDroZwg7KCB7Ja0"
           "IOuRkOuKlCDqsoPsnoXri4jri6QuIOyFi+ynuCDrgqAgU0FSSU1BIOyYiOy4oe2RnChsbzk1wrdoaTk1KeydmCDsirXqtIDrjIDroZwg7KCB"
           "7Jy866m0IOyVvSA5MCUg66W8IOOAjDk1JSDrs7Tri6Qg66qo7J6Q652864uIIOyigeuLpOOAjeuhnCDqsbDqvrjroZwg7J297Iq164uI64uk"
           "LiBpbnRlcnZhbF93aWR0aCDrpbwg7J297Ja0IOy2nOugpe2VmOuptCDsnbQg7Iuk7IiY6rCAIOyDneq4sOyngCDslYrsirXri4jri6QuCi0g"
           "7Z2U7ZWcIOyLpOyImCDrkZjsnYAg7ZWcIOyjvOydmCDslb0gOTAlIOunjCDrs7Tqs6AgaW50ZXJ2YWxfd2lkdGgg66W8IOuCruy2lOuKlCDq"
           "soPsnoXri4jri6QuIOqwmeydgCDqtazshLHsnLzroZwg7JiI7LihIOybkOygkCA16rCc66W8IOqzhOyCsO2VmOuptCDsi6TsoJwg7Y+s7ZWo"
           "66Wg7J20IOyVvSA4MO+9njkwJSwg7Y+J6regIOyVvSA4NSUg7J206rOgIDEx7JuUIDE27J28IOybkOygkOydtCDqsIDsnqUg64aS7Iq164uI"
           "64ukLiAxMOyblCAxOeydvMK3Mjbsnbzqs7wgMTHsm5QgMuydvCDsm5DsoJDsnYAg67KX7Ja064KcIOyLnOqwgeydtCDso7zroZwg6rWs6rCE"
           "IOyVhOuemOyYgOyKteuLiOuLpC4KLSDsnbTrsogg7KO87JeQIO2VoCDsnbw6IOyatOyYgSDsmIjsuKHsnYAg6rOE7KCIIOuCmOydtOu4jCAx"
           "Njgg66GcIOuRoeuLiOuLpC4gUHJvcGhldCDqtazqsITsnYQg7LC46rOg66GcIO2VqOq7mCDrs7Tqs6DtlaAg65WM64qUIOOAjDgwJSDsmIjs"
           "uKHqtazqsIQsIDEx7JuUIDE3772eMjPsnbwg7Iuk7KCcIO2PrO2VqOuloCDslb0gOTAlLCDtj4nqt6Ag6rWs6rCEIOq4uOydtCDslb0gMSww"
           "MzDrjIDjgI3sspjrn7wg7IS4IOqwgOyngOulvCDtlajqu5gg7KCB7Iq164uI64ukLiBpbnRlcnZhbF93aWR0aCDripQg6riw67O46rCSIDAu"
           "ODAg6re464yA66GcIOuRkOqzoCwg67CU6r+A7KeA64qUIOyYiOy4oSDsm5DsoJAgNeqwnOydmCDtj6ztlajrpaDsnYQg7ZWcIO2RnOuhnCDr"
           "qqjsnYAg65KkIOygle2VqeuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM65Oc64qUIOuwqeyLnS4gbS5pbnRlcnZhbF93aWR0aCDsvZTr"
           "k5wg7ZWcIO2WieydtCDruYTqtZAg6riw7KSA7J2EIOuqqO2YleyXkOyEnCDsnb3slrQg7Ji164uI64ukLiBpbnNpZGUg66W8IOunjOuTnOuK"
           "lCDsvZTrk5wg7ZWcIO2WieydtCAxNjjsi5zqsIHrp4jri6Qg7Iuk7Lih7J20IO2VmO2VnCDsnbTsg4HsnbTqs6Ag7IOB7ZWcIOydtO2VmOyd"
           "uOyngCDssLjCt+qxsOynk+ydhCDrp4zrk6Tqs6AsIOq3uCDtj4nqt6DsnbQg7Iuk7KCcIO2PrO2VqOuloOyeheuLiOuLpC4g7JiI7LihIOyn"
           "geyghOyXkCDsi5zrk5zrpbwg6rOg7KCV7ZW07JW8IO2RnOuzuCDqsr3roZzqsIAg6rCZ7JWE7KC4IOydtCDqsJLsnbQg64uk7IucIOuCmOyY"
           "teuLiOuLpC4g7KCB7ZWp7J2AIOuCnOyImOulvCDsk7Dsp4Ag7JWK7JWELCDsoIHtlakg7KeB7KCEIOqzoOygleydgCDssKjroYDrpbwg66ee"
           "7LaU64qUIOqyg+yeheuLiOuLpC4K"),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuT"
           "pOyWtOyguCDsnojripQgdHJhaW4sIHZhbGlkLCBob2xpZGF5c19vbmUsIFJFR1MsIEJBU0VfUk1TRSwgU05BSVZFX1JNU0UsIFNFRUQsIHJt"
           "c2Ug66W8IOq3uOuMgOuhnCDsk7Dqs6Ag7IOI66GcIOunjOuTpOyngOuKlCDrp5DslYQKICAgIOykmC4g7ZWZ7Iq1IO2RnOuKlCB0cmFpbiDs"
           "nZgg7J24642x7Iqk66W8IGRzLCBkZW1hbmQg66W8IHkg66GcIOyYruq4sOqzoCBSRUdTIOuRkCDsl7QgcmFpbsK3dGVtcCDrpbwg7ZWo6ruY"
           "IOuLtOqzoCwg7JiI7LihIO2RnOuKlCB2YWxpZCDsnZgg7J24642x7Iqk66W8CiAgICBkcyDroZwg65GQ6rOgIOqwmeydgCDrkZAg7Je07JeQ"
           "IOqygOymnSDqtazqsIQg7Iuk7Lih7J2EIOuLtOyVhCDspJguIFByb3BoZXQg7J2AIOuvuOyFmCAxIOqzvCDqsJnqsowg7Jew6rCEIOqzhOyg"
           "iOyEseydgCDsoJzsmbjtlZjqs6Ag7KO86rCEwrfsnbzqsIQg6rOE7KCI7ISx7J2AIO2PrO2VqO2VmOqzoCwKICAgIGhvbGlkYXlzX29uZSDt"
           "nLTsnbzroZwg66eM65Ok6rOgIFJFR1Mg66W8IGFkZF9yZWdyZXNzb3Ig66GcIOuNlO2VtCDspJguIOygge2VqSDsp4HsoITqs7wg7JiI7Lih"
           "IOyngeyghOyXkCBucC5yYW5kb20uc2VlZChTRUVEKSDrpbwg67aI65+sIOykmC4g6rOE7IiY64qUCiAgICBwcm9waGV0LnV0aWxpdGllcyDs"
           "nZggcmVncmVzc29yX2NvZWZmaWNpZW50cyDroZwg7JuQ656YIOuLqOychCjrjIAv64+EwrfrjIAvbW0p66GcIOydveyWtCDspJguICLrgqDs"
           "lKjrpbwg642U7ZWcIOqygOymnSAxNjjsi5zqsIQgUk1TRSA9IOqwkiDrjIAiLCAi6riw7JioIOqzhOyImCA9CiAgICDqsJIg64yAL+uPhCIs"
           "ICLqsJXsiJgg6rOE7IiYID0g6rCSIOuMgC9tbSIg66W8IOyGjOyImCDrkZjsp7gg7J6Q66as66GcIO2VnCDtlonslKkgcHJpbnQg7ZWY6rOg"
           "LCDrgqDslKjrpbwg642U7ZWcIFJNU0Ug6rCAIEJBU0VfUk1TRSDrs7Tri6Qg7J6R7J2EIOuVjOunjAogICAg642U7ZWc64uk64qUIO2MkOuL"
           "qCDtlZwg7ZaJ64+EIOywjeyWtCDspJguIOq3uOuemO2UhOuKlCBTTkFJVkVfUk1TRSwgQkFTRV9STVNFLCDrgqDslKjrpbwg642U7ZWcIFJN"
           "U0Ug66W8IOOAjOqzhOygiCDrgpjsnbTruIzjgI3jgIzrgqDslKgg7JeG7J2M44CN44CM64Kg7JSoCiAgICDsnojsnYzjgI0g6rCA66GcIOun"
           "ieuMgCDshYvsnLzroZwg6re466as6rOgIOqwgOuhnOy2lSDsnbTrpoTsnYQg64us7JWEIOykmC4KCiAgICBmcm9tIHByb3BoZXQudXRpbGl0"
           "aWVzIGltcG9ydCByZWdyZXNzb3JfY29lZmZpY2llbnRzCgogICAgdHIgPSBwZC5EYXRhRnJhbWUoeyJkcyI6IHRyYWluLmluZGV4LCAieSI6"
           "IHRyYWluWyJkZW1hbmQiXS52YWx1ZXMsCiAgICAgICAgICAgICAgICAgICAgICAgInJhaW4iOiB0cmFpblsicmFpbiJdLnZhbHVlcywgInRl"
           "bXAiOiB0cmFpblsidGVtcCJdLnZhbHVlc30pCiAgICBmdXQgPSBwZC5EYXRhRnJhbWUoeyJkcyI6IHZhbGlkLmluZGV4LCAgICAgICAgICAg"
           "ICAgICAgICAgIyDsmIjsuKEg7ZGc7JeQ64+EIOqwmeydgCDrkZAg7Je07J20IOyeiOyWtOyVvCDtlZzri6QKICAgICAgICAgICAgICAgICAg"
           "ICAgICAgInJhaW4iOiB2YWxpZFsicmFpbiJdLnZhbHVlcywgInRlbXAiOiB2YWxpZFsidGVtcCJdLnZhbHVlc30pCgogICAgbnAucmFuZG9t"
           "LnNlZWQoU0VFRCkKICAgIG0gPSBQcm9waGV0KHllYXJseV9zZWFzb25hbGl0eT1GYWxzZSwgd2Vla2x5X3NlYXNvbmFsaXR5PVRydWUsCiAg"
           "ICAgICAgICAgICAgICBkYWlseV9zZWFzb25hbGl0eT1UcnVlLCBob2xpZGF5cz1ob2xpZGF5c19vbmUpICAgIyDrr7jshZggMSDqs7wg6rCZ"
           "7J2AIOq1rOyEsQogICAgZm9yIHIgaW4gUkVHUzoKICAgICAgICBtLmFkZF9yZWdyZXNzb3IocikgICAgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgICAgICAjIOyXtCDtlZjrgpjsl5Ag6rOE7IiYIO2VmOuCmAogICAgbS5maXQodHIpCiAgICBucC5yYW5kb20uc2VlZChTRUVEKQog"
           "ICAgcCA9IG0ucHJlZGljdChmdXQpCgogICAgY29lZiA9IHJlZ3Jlc3Nvcl9jb2VmZmljaWVudHMobSkgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICMg7ZGc7KSA7ZmU7ZW0IOy2lOygle2VnCDqs4TsiJjrpbwg7JuQ656YIOuLqOychOuhnCDrs7Xsm5DtlZzri6QKICAgIGNfdGVtcCA9"
           "IGZsb2F0KGNvZWYubG9jW2NvZWZbInJlZ3Jlc3NvciJdID09ICJ0ZW1wIiwgImNvZWYiXS5pbG9jWzBdKQogICAgY19yYWluID0gZmxvYXQo"
           "Y29lZi5sb2NbY29lZlsicmVncmVzc29yIl0gPT0gInJhaW4iLCAiY29lZiJdLmlsb2NbMF0pCiAgICByZWdfcm1zZSA9IHJtc2UodmFsaWRb"
           "ImRlbWFuZCJdLCBwWyJ5aGF0Il0pCiAgICBwcmludChmIuuCoOyUqOulvCDrjZTtlZwg6rKA7KadIDE2OOyLnOqwhCBSTVNFID0ge3JlZ19y"
           "bXNlOi4yZn0g64yAIikKICAgIHByaW50KGYi6riw7JioIOqzhOyImCA9IHtjX3RlbXA6Ky4yZn0g64yAL+uPhCIpCiAgICBwcmludChmIuqw"
           "leyImCDqs4TsiJggPSB7Y19yYWluOisuMmZ9IOuMgC9tbSIpCiAgICBpZiByZWdfcm1zZSA8IEJBU0VfUk1TRToKICAgICAgICBwcmludCgi"
           "7YyQ64uoOiDrgqDslKgg7Je07J2EIOuNlO2VnOuLpC4g7Jq07JiBIOyYiOy4oSDtkZzsnZgg6riw7JiowrfqsJXsiJjripQg6riw7IOB7LKt"
           "IOyYiOuztOuhnCDssYTsmrTri6QiKQogICAgZWxzZToKICAgICAgICBwcmludCgi7YyQ64uoOiDsnbTrsogg6rKA7KadIOq1rOqwhOyXkOuK"
           "lCDrgqDslKgg7Je07J2EIOuNlO2VmOyngCDslYrripTri6QuIOyatOyYgeyXkOyEnCDsk7gg65WM64qUIOq4sOyDgeyyrSDsmIjrs7TroZwg"
           "7LGE7Jq064ukIikKCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XLCAzLjApKQogICAgYXguYmFyaChbMCwgMSwg"
           "Ml0sIFtTTkFJVkVfUk1TRSwgQkFTRV9STVNFLCByZWdfcm1zZV0sIGhlaWdodD0wLjU1LAogICAgICAgICAgICBjb2xvcj1bQ09MWyLsl7Dt"
           "mozsg4kiXSwgQ09MWyLtmozsg4kiXSwgQ09MWyLtlZjripjsg4kiXV0pCiAgICBheC5zZXRfeXRpY2tzKFswLCAxLCAyXSkKICAgIGF4LnNl"
           "dF95dGlja2xhYmVscyhbIuqzhOygiCDrgpjsnbTruIwiLCAi64Kg7JSoIOyXhuydjCIsICLrgqDslKgg7J6I7J2MIl0pCiAgICBheC5zZXRf"
           "eGxhYmVsKCLqsoDspp0gMTY47Iuc6rCEIFJNU0Uo64yAKSIpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0KCi0g7JiI7IOBIOyFiyDqsIDsmrTr"
           "jbAg64u17J2AIOyEuCDrsojsp7jsnoXri4jri6QuIOqygOymnSAxNjjsi5zqsIQgUk1TRSDqsIAg64Kg7JSoIOyXhuuKlCDqtazshLHsnZgg"
           "7JW9IDMxMOuMgOyXkOyEnCDslb0gMzQw64yA66GcIOy7pOyhjOqzoCwg6rOE7KCIIOuCmOydtOu4jCAxNjgg7J2YIDIwNS4yNeuMgOyZgOuK"
           "lCDrjZQg66mA7Ja07KGM7Iq164uI64ukLiDqs4TsiJjsnZgg67aA7Zi464qUIOyDgeyLneqzvCDrp57sirXri4jri6QuIOq4sOyYqCDqs4Ts"
           "iJjripQg7JW9ICs0MOuMgC/rj4QsIOqwleyImCDqs4TsiJjripQg7JW9IOKIkjEyNOuMgC9tbSDsnoXri4jri6QuCi0g7Luk7KeEIOq5jOuL"
           "reydgCDqs4TsiJjqsIAg6rOx7ZW07KeA64qUIOqwkuyXkCDsnojsirXri4jri6QuIOq4sOyYqCDtla3snYAg6riw7JioIOqzhOyImCDDlyAo"
           "6re4IOyLnOqwgSDquLDsmKgg4oiSIO2VmeyKtSDtj4nqt6AgMTUuODPrj4Qp7J206rOgLCDqsoDspp0g6rWs6rCEIO2Pieq3oCDquLDsmKjs"
           "nbQgNC45MeuPhOudvCDsnbQg7ZWt7J20IOyYiOy4oeydhCDsi5zqsITri7kg7Y+J6regIOyVvSA0MznrjIAg64K066C47Iq164uI64ukLiDq"
           "sJXsiJgg7ZWt6rO8IOuLpOyLnCDsoIHtlanrkJwg7LaU7IS46rCAIOydvOu2gOulvCDsg4Hsh4TtlbQsIOyYiOy4oSDtj4nqt6DsnYAg7JW9"
           "IDcwN+uMgOyXkOyEnCDslb0gNTcx64yA66GcIOyLpOy4oSDtj4nqt6AgNjk0LjI064yA67O064ukIOuCruyVhOyhjOyKteuLiOuLpC4g7ZWZ"
           "7Iq1IOq1rOqwhOydmCDsi5zqsITri7kg7Y+J6regIOyImOyalOuKlCAw772eNeuPhCA2NjLrjIAsIDXvvZ4xMOuPhCA2NznrjIDroZwg6rGw"
           "7J2YIOqwmeydgOuNsCwg7KeB7ISgIOq4sOyYqCDtla3snYAgNeuPhOuniOuLpCDslb0gMjAx64yA66W8IOu6jeuLiOuLpC4g6rKA7KadIDE2"
           "OOyLnOqwhCDqsIDsmrTrjbAgMTAx7Iuc6rCE7J20IO2VmeyKtSDquLDsmKjsnZggNSUg7KeA7KCQIDYuMuuPhOuztOuLpCDstpTsm6DsirXr"
           "i4jri6QuCi0g7Z2U7ZWcIOyLpOyImCDtlZjrgpjripQg6rOE7IiY66W8IOq4sOyYqCDsm5DqsJLsl5Ag6rOx7ZW0IDQwIMOXIDQuOTEg4omI"
           "ICsxOTfrjIDroZwg7J2964qUIOqyg+yeheuLiOuLpC4g7Z2U7ZWcIOyLpOyImCDrkZjsnYAg7JW9IDM0MOuMgOulvCDsmrTsmIEg7ISx64ql"
           "7Jy866GcIOyggeuKlCDqsoPsnoXri4jri6QuIOyYiOy4oSDtkZzsl5Ag7Iuk7LihIOuCoOyUqOulvCDrjIDsnoXtlZwg7IKs7ZuEIOyYiOy4"
           "oeydtOudvCwg7Jq07JiB7JeQ7IScIOyYiOuztOulvCDrjIDsnoXtlZjrqbQg7JiI67O06rCAIDHrj4Qg67mX64KY6rCIIOuVjOuniOuLpCDs"
           "mIjsuKHsnbQg7JW9IDQw64yAIOuNlCDruZfrgpjqsJHri4jri6QuCi0g66+4656YIOqwkuydhCDssYTsmrgg7IiYIOyeiOuKlOqwgDog6riw"
           "7JiowrfqsJXsiJjripQg6riw7IOB7LKtIOyYiOuztOuhnCDri6TsnYwgN+ydvCDqsJLsnYQg7LGE7Jq4IOyImCDsnojslrQg7Jm47IOdIOuz"
           "gOyImCDtm4Trs7TripQg66ee7Iq164uI64ukLiDri6Trp4wg7J2067KIIOqygOymnSDqtazqsITsspjrn7wg7JiI7LihIOq4sOqwhCDquLDs"
           "mKjsnbQg7ZWZ7Iq1IOuylOychCDrsJbsnbTrqbQg642U7ZWY7KeAIOyViuyKteuLiOuLpC4KLSDsnbTrsogg7KO87JeQIO2VoCDsnbw6IOya"
           "tOyYgSDsmIjsuKHsnYAg6rOE7KCIIOuCmOydtOu4jCAxNjgg6re464yA66GcIOuRkOqzoCDrgqDslKgg7Je07J2AIOuNlO2VmOyngCDslYrs"
           "irXri4jri6QuIOuztOqzoOyEnOyXkOuKlCDjgIzrgqDslKgg7Y+s7ZWoIOyVvSAzNDDrjIDripQg7Iuk7LihIOuCoOyUqOulvCDrjIDsnoXt"
           "lZwg7IKs7ZuEIOyYiOy4oeOAjeydtOudvOqzoCDsoIHsirXri4jri6QuIOuCoOyUqOulvCDri6Tsi5wg7Iuc64+E7ZWgIOuVjOuKlCDsmIjs"
           "uKEg6riw6rCE7JeQIO2VmeyKtSA1JSDsp4DsoJDrs7Tri6Qg7LaU7Jq0IOyLnOqwgeydtCDrqocg7Iuc6rCE7J247KeA67aA7YSwIO2Zleyd"
           "uO2VqeuLiOuLpC4g6rCV7IiY66eMIOuNlO2VnCDqtazshLHsnYAg7JW9IDMxMOuMgOuhnCDrgqDslKgg7JeG64qUIOq1rOyEseqzvCDqsbDs"
           "nZgg6rCZ7JWELCDsl7TsnYAg7ZWcIOuyiOyXkCDtlZjrgpjslKkg642U7ZW0IOu5hOq1kO2VqeuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg"
           "66eM65Oc64qUIOuwqeyLnS4g7KeA7ZGc6rCS7J2EIOuwlOq+uOuKlCDqsoPsnYAgYWRkX3JlZ3Jlc3NvciDrkZAg67KI6rO8LCDtlZnsirUg"
           "7ZGcwrfsmIjsuKEg7ZGcIOyWkeyqveyXkCDqsJnsnYAg65GQIOyXtOydhCDssYTsmrQg7L2U65Oc7J6F64uI64ukLiByZWdyZXNzb3JfY29l"
           "ZmZpY2llbnRzKG0pIOy9lOuTnCDtlZwg7ZaJ7J20IO2RnOykgO2ZlO2VtCDstpTsoJXtlZwg6rOE7IiY66W8IOuMgC/rj4TCt+uMgC9tbSDr"
           "oZwg67O17JuQ7ZWp64uI64ukLgo="),
}
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6eyLrqoXrqqkg7Y+s7ZWo66WgIjpbIjk1IOuCmCA4MCDsnYQg7L2U65Oc7JeQIOyIq+yekOuhnCDsoIHsp4Ag66eQ6rOgLCDsoIHt"
           "lantlZwgUHJvcGhldCDrqqjtmJXsnZggaW50ZXJ2YWxfd2lkdGgg66W8IOydveyWtCAlIOuhnCDstpzroKXtlZjrnbzqs6AgUFJPTVBUIOyX"
           "kCDsoIHsnLzshLjsmpQuIOOAjOuqheuqqSDtj6ztlajrpaAgPSDsiKvsnpAl44CNIO2VnCDtlonsnoXri4jri6QiLCJpbnRlcnZhbF93aWR0"
           "aCDripQgMC44IOqwmeydgCDruYTsnKjsnbTrnbwgMTAwIOydhCDqs7HtlZjqsbDrgpggOi4wJSDtmJXsi53snLzroZwg7LCN7Ja07JW8ICUg"
           "6rCS7J20IOuQqeuLiOuLpC4g7Iir7J6Q66W8IOyGkOycvOuhnCDsoIHsnLzrqbQg66qo7ZiV6rO8IOuLpOuluCDquLDspIDsnLzroZwg67mE"
           "6rWQ7ZWY6rKMIOuQqeuLiOuLpCIsIuydtCDsvZTrk5zqsIAg66qF66qpIO2PrO2VqOuloOydhCDslrTrlJTshJwg6rCA7KC47JmU64qU6rCA"
           "Il0sIuyLpOygnCDtj6ztlajrpaAiOlsi7Iuk7Lih7J20IHloYXRfbG93ZXIg7J207IOB7J206rOgIHloYXRfdXBwZXIg7J207ZWY7J24IOyL"
           "nOqwgeydmCDruYTsnKjsnoXri4jri6QuIHZhbGlkIDE2OOyLnOqwgeyXkOyEnCDqs4TsgrDtlbQgJSDroZwsIDE2OOyLnOqwhCDqsIDsmrTr"
           "jbAg66qHIOyLnOqwhOyduOyngOyZgCDtlajqu5gg44CM7Iuk7KCcIO2PrO2VqOuloCA9IOyIq+yekCXjgI0g7ZWcIO2WieycvOuhnCDstpzr"
           "oKXtlZjshLjsmpQiLCLsoIHtlakg7KeB7KCE6rO8IOyYiOy4oSDsp4HsoITsl5AgbnAucmFuZG9tLnNlZWQoU0VFRCkg66W8IOu2iOuggOuK"
           "lOyngCwg7Jew6rCEIOqzhOygiOyEseydgCDrubzqs6Ag7KO86rCEwrfsnbzqsIQg6rOE7KCI7ISx6rO8IGhvbGlkYXlzX29uZSDsnYAg64Sj"
           "7JeI64qU7KeAIO2ZleyduO2VmOyEuOyalC4g6rWs6rCE7J2AIO2RnOuzuCDqsr3roZzroZwg66eM65Ok7Ja0IOyLnOuTnOyZgCDqtazshLHs"
           "nbQg64uk66W066m0IOqwkuydtCDri6zrnbzsp5Hri4jri6QiLCLsnbQg7L2U65Oc6rCAIOq1rOqwhCDslYjsnbjsp4Drpbwg7Ja064qQIOuR"
           "kCDsl7TroZwg7YyQ64uo7ZaI64qU6rCAIl0sInBsb3QiOlsi7ZWcIOq3uOumvOyXkCDsi6TsuKHshKDqs7wg7JiI7Lih7ISg7J20IOqyueyz"
           "kCDsnojripTsp4Agwrcg6rWs6rCE7J2EIOyYheqyjCDsuaDtlojripTsp4Agwrcg7ISg66eI64ukIOygkOydtCAxNjjqsJzsnbjsp4Agwrcg"
           "6rCA66Gc7LaVwrfshLjroZzstpXsl5Ag7J2066aE7J2EIOuLrOyVmOuKlOyngCDssKjroYDroZwg7ZmV7J247ZWY7IS47JqULiDqt7jrprzs"
           "nYAg6rCS6rO8IOqwmeydgCBQUk9NUFQg7JeQIO2VqOq7mCDsmpTssq3tlanri4jri6QiLCLrgrQg6re466a87J2YIOyWtOuKkCDstpXCt+yW"
           "tOuKkCDshKDsnbQg6riw7KSA6rO8IOyWtOq4i+uCrOuKlOyngCJdLCJlcnJvciI6WyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA"
           "66GcIO2KnO2EsOyXkCDrs7Tsl6wg7KO86rOgLCDjgIzrr7jshZggMSDspIDruYQg7IWA44CN7J20IOunjOuToCDsnbTrpoQodHJhaW4gwrcg"
           "dmFsaWQgwrcgaG9saWRheXNfb25lKeqzvCBTRUVEIOuKlCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7JOw65286rOgIOu2"
           "hOuqhe2eiCDsoIHsnLzshLjsmpQiLCLsnbQg7Jik66WY6rCAIOyWtOuKkCDsvZTrk5wg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX0sIm0y"
           "Ijp7IuuCoOyUqOulvCDrjZTtlZwg6rKA7KadIDE2OOyLnOqwhCBSTVNFIjpbIuuvuOyFmCAxIOqzvCDri6zrnbzsp4Qg6rOz7J20IOuRkCDs"
           "l7Trv5Dsnbjsp4Ag7ZmV7J247ZWY7IS47JqULiDsl7DqsIQg6rOE7KCI7ISx7J2AIOu5vOqzoCDso7zqsITCt+ydvOqwhCDqs4TsoIjshLHq"
           "s7wgaG9saWRheXNfb25lIOydgCDrhKPsnYAg6rWs7ISx7J2EIOq3uOuMgOuhnCDrkZDqs6AsIO2VmeyKtSDtkZzsmYAg7JiI7LihIO2RnCDs"
           "lpHsqr3sl5AgcmFpbsK3dGVtcCDsl7TsnYQg64u07JWY64qU7KeAIOu0heuLiOuLpC4g6rCS7J2AIOOAjOydtOumhCA9IOyIq+yekCDrjIDj"
           "gI0g7ZWcIO2WieycvOuhnCDstpzroKXtlanri4jri6QiLCLsmIjsuKEg7ZGc7JeQIOuMgOyehe2VnCDquLDsmKjCt+qwleyImOqwgCB2YWxp"
           "ZCDsnZgg7Iuk7Lih7J247KeALCDsoIHtlakg7KeB7KCE6rO8IOyYiOy4oSDsp4HsoITsl5Ag7Iuc65Oc66W8IOqzoOygle2WiOuKlOyngCBQ"
           "Uk9NUFQg7JeQIOyggeycvOyEuOyalC4g7J6Q66a/7IiY64qUIOyGjOyImCDrkZjsp7gg7J6Q66as7J6F64uI64ukIiwi7J20IOy9lOuTnOqw"
           "gCDrr7jshZggMSDsnZgg6rWs7ISx7JeQ7IScIOuwlOq+vCDqs7PsnbQg7Ja065SU7J246rCAIl0sIuq4sOyYqCDqs4TsiJgiOlsi6rOE7IiY"
           "66W8IOyGkOycvOuhnCDrs7Xsm5DtlZjsp4Ag66eQ6rOgLCDsoIHtlantlZwg66qo7ZiV7J2EIHJlZ3Jlc3Nvcl9jb2VmZmljaWVudHMg7JeQ"
           "IOuEmOqyqCB0ZW1wIO2WieydmCBjb2VmIOulvCDqs6jrnbwg7Jik65286rOgIOyggeycvOyEuOyalC4g67aA7Zi46rmM7KeAIOOAjOq4sOyY"
           "qCDqs4TsiJggPSDsiKvsnpAg64yAL+uPhOOAjSDtlZwg7ZaJ7Jy866GcIOy2nOugpe2VqeuLiOuLpCIsIuydtCDsvZTrk5zqsIAg6rOE7IiY"
           "66W8IOybkOuemCDri6jsnIQo64yAL+uPhCnroZwg67O17JuQ7ZWcIOqzs+ydtCDslrTrlJTsnbjqsIAiXSwicGxvdCI6WyLqsIDroZwg66eJ"
           "64yA6rCAIOyFi+yduOyngCDCtyDshLjroZwg64iI6riIIOydtOumhOydtCDjgIzqs4TsoIgg64KY7J2067iM44CNwrfjgIzrgqDslKgg7JeG"
           "7J2M44CNwrfjgIzrgqDslKgg7J6I7J2M44CN7J247KeAIMK3IOqwgOuhnOy2lSDsnbTrpoTsnYQg64us7JWY64qU7KeAIO2ZleyduO2VmOyE"
           "uOyalC4g7JWeIOuRkCDrp4nrjIDripQgU05BSVZFX1JNU0Ug7JmAIEJBU0VfUk1TRSDrpbwg6re464yA66GcIOyUgeuLiOuLpCIsIuuCtCDq"
           "t7jrprzsnZgg66eJ64yAIOyImMK36ri47J206rCAIOq4sOykgOqzvCDslrTquIvrgpwg6rOz7J20IOyWtOuUlOyduOyngCJdLCJlcnJvciI6"
           "WyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIO2KnO2EsOyXkCDrs7Tsl6wg7KO86rOgLCDjgIzrr7jshZggMiDspIDruYQg"
           "7IWA44CN7J20IOunjOuToCDsnbTrpoQodHJhaW4gwrcgdmFsaWQgwrcgaG9saWRheXNfb25lIMK3IFJFR1MgwrcgQkFTRV9STVNFIMK3IFNO"
           "QUlWRV9STVNFKeydgCDri6Tsi5wg66eM65Ok7KeAIOunkOudvOqzoCDsoIHsnLzshLjsmpQuIOOAjFJlZ3Jlc3NvciAuLi4gbWlzc2luZ+OA"
           "jSDsmKTrpZjripQg7JiI7LihIO2RnOyXkCDrkZAg7Je07J20IOu5oOyhjOuLpOuKlCDrnLvsnoXri4jri6QiLCLsnbQg7Jik66WY6rCAIOyW"
           "tOuKkCDsvZTrk5wg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX19")
_ASK = {"m1": "full", "m2": "blank"}   # 미션 2 는 튜터에 보낼 질문의 빈칸을 내가 채웁니다

_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 항목 틀 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 점검할 항목 틀 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 코드 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 출력하면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 수직선이 「수직선이 아예 없다」 로 판단돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax, min_pts=10):
    """축의 선을 곡선(점 `최소점` 개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다.

    문턱을 인자로 연 것은 오늘 그림의 선이 **점 일곱 개**(검증 7일)이기 때문입니다. 문턱을
    10 으로 고정해 두면 규격대로 그린 학생의 선이 「곡선이 0개다」 로 읽혀 고칠 곳 없는
    결함을 고치러 가게 됩니다.
    """
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= min_pts:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준값의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax, spec.get("min_points", 10))
        n_spec = spec.get("n_series", len(cur))      # 정수면 딱 그 개수, 두 값이면 그 사이
        n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
        if not (n_lo <= len(cur) <= n_hi):
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            gaps.append(f"선이 {len(cur)}개다 (기준 {ref_text})")
        if cur and "n_points" in spec:
            n_spec = spec["n_points"]                   # 정수면 딱 그 개수, 두 값이면 그 사이
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            off_range = [int(x.size) for x, _ in cur if not (n_lo <= x.size <= n_hi)]
            if off_range:
                ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
                gaps.append(f"점이 {off_range[0]}개인 선이 있다 (기준 {ref_text})")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도면 봉우리가 그 자리에 섭니다
                trough = float(xd[yd.argmin()])         # NLL 로 그렸으면 골짜기가 그 자리입니다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(반응 곡선) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("최댓값 위치를 가리키는 세로선이 기준 구간에 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec:
            n_spec = spec["n"]                          # 정수면 딱 그 개수, 두 값이면 그 사이(예: 시차 24~400)
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            if not (n_lo <= len(heights) <= n_hi):
                gaps.append(f"막대가 {len(heights)}개다 (기준 {ref_text})")
        elif not heights:                              # 막대 수 기준이 없는 경우에도 빈 축은 통과시키지 않습니다
            gaps.append("막대가 하나도 없다 (이 축에서 막대를 찾지 못했습니다)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "barh":
        width = [float(p.get_width()) for p in ax.patches]   # 가로 막대는 **길이**가 값입니다
        if "n" in spec:
            n_spec = spec["n"]                          # 정수면 딱 그 개수, 두 값이면 그 사이
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            if not (n_lo <= len(width) <= n_hi):
                gaps.append(f"가로 막대가 {len(width)}개다 (기준 {ref_text})")
        elif not width:
            gaps.append("가로 막대가 하나도 없다 (이 축에서 막대를 찾지 못했습니다)")
        expected = spec.get("막대값", [])
        if width and expected and max(width) * 4 < min(lo for lo, _ in expected):
            gaps.append("막대 길이가 모두 값보다 훨씬 작다 — 세로 막대는 길이가 아니라 굵기가"
                        " 값이 되니, 값이 길이로 읽히는 가로 막대로 그립니다")
        else:
            for lo, hi in expected:
                if any(lo <= v <= hi for v in width):
                    continue
                now = ", ".join(f"{v:,.1f}" for v in width[:4]) if width else "막대 없음"
                gaps.append(f"길이가 {lo:g}~{hi:g} 인 막대가 없다 (지금 {now})")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 컬렉션(LineCollection)은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 가로축 눈금 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("가로축라벨") and not ax.get_xlabel().strip():
        gaps.append("가로축 라벨이 비어 있다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림. 「미션 1 준비 셀」의 train·valid 로 실측·예측·예측구간을 겹칩니다"""
    train_, valid_ = _get("train"), _get("valid")
    hol_, Prophet_ = _get("holidays_one"), _get("Prophet")
    if train_ is None or valid_ is None or Prophet_ is None:
        return None
    seed_v = _get("SEED") or 42
    np.random.seed(seed_v)                           # 구간은 표본 경로라 두 번 고정합니다
    m = Prophet_(yearly_seasonality=False, weekly_seasonality=True,
                 daily_seasonality=True, holidays=hol_)
    m.fit(pd.DataFrame({"ds": train_.index, "y": train_["demand"].values}))
    np.random.seed(seed_v)
    p = m.predict(pd.DataFrame({"ds": valid_.index}))
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.fill_between(valid_.index, p["yhat_lower"], p["yhat_upper"],
                   color=COL["하늘색"], alpha=0.22)
    a.plot(valid_.index, valid_["demand"], color=COL["남색"], lw=1.0)
    a.plot(valid_.index, p["yhat"], color=COL["하늘색"], lw=1.2)
    peak = float(valid_["demand"].max())
    a.text(valid_.index[6], peak * 0.98, "실측", color=COL["남색"])
    a.text(valid_.index[96], peak * 0.98, "예측과 80% 예측구간", color=COL["하늘색"])
    a.set_xlabel("검증 168시간")
    a.set_ylabel("수요(대)")
    return f


def _ref_plot_m2():
    """미션 2 의 참조 그림. 「미션 2 준비 셀」의 값들로 세 RMSE 를 가로 막대 셋으로 그립니다"""
    train_, valid_ = _get("train"), _get("valid")
    hol_, Prophet_ = _get("holidays_one"), _get("Prophet")
    prev_score = [_get("SNAIVE_RMSE"), _get("BASE_RMSE")]
    if train_ is None or valid_ is None or Prophet_ is None or None in prev_score:
        return None
    seed_v = _get("SEED") or 42
    train_tbl = {"ds": train_.index, "y": train_["demand"].values}
    pred_tbl = {"ds": valid_.index}
    np.random.seed(seed_v)
    m = Prophet_(yearly_seasonality=False, weekly_seasonality=True,
                 daily_seasonality=True, holidays=hol_)
    for r in (_get("REGS") or ["rain", "temp"]):
        m.add_regressor(r)                       # 회귀변수는 양쪽 표에 열이 있어야 합니다
        train_tbl[r] = train_[r].values
        pred_tbl[r] = valid_[r].values
    m.fit(pd.DataFrame(train_tbl))
    np.random.seed(seed_v)
    val = list(prev_score) + [rmse(valid_["demand"], m.predict(pd.DataFrame(pred_tbl))["yhat"])]
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.0))
    a = f.add_subplot(111)
    a.barh([0, 1, 2], val, height=0.55,
           color=[COL["연회색"], COL["회색"], COL["하늘색"]])
    a.set_yticks([0, 1, 2])
    a.set_yticklabels(["계절 나이브", "날씨 없음", "날씨 있음"])
    for i, v in enumerate(val):
        a.text(v + 6, i, f"{v:,.1f}", va="center", color=COL["남색"])
    a.set_xlim(0, max(val) * 1.25)
    a.set_xlabel("검증 168시간 RMSE(대)")
    return f


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2}   # 미션마다 참조 그림을 그리는 함수 하나


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing():
    """PROMPT 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return "PROMPT 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return "PROMPT 입력란이 자리표시자·견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 채점 없이 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                        # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason:
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 코드 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 출력한 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 출력하면 채점이 어느 숫자가 답인지 선별하지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        hits = [v for v in (_nums("\n".join(tail)) if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            need_axes = int(plot_spec.get("n_axes", 1))   # 그림 두 장을 나란히 요구하는 기준이면 2 입니다
            ok_axes, gaps = 0, None
            for ax in axs:                 # 기준에 맞는 축이 필요한 수만큼 있으면 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    ok_axes += 1
                    if ok_axes >= need_axes:
                        break
                elif gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if ok_axes < need_axes:
                passed = False
                failed.append(("그래프", "plot"))
                extra_msgs = [f"기준에 맞는 그림이 {ok_axes}개다 (기준 {need_axes}개)"] if need_axes > 1 else []
                print("❌ 그래프 — " + " · ".join(extra_msgs + (gaps or [])))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다. PROMPT 에 시킬 일을 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 서울 공유자전거 준비

자료는 **서울 공유자전거 시간당 대여 기록**(UCI 머신러닝 저장소)입니다. 2017년 12월 1일부터 2018년 11월 30일까지 8,760행입니다. 휴무 295시간을 1주일 전 같은 시각 값으로 채운 보정 수요 `demand` 를 씁니다. **Prophet 기본 구성**은 연간 계절성을 제외하고 주간·일간 계절성과 휴일 데이터프레임(공휴일 18일)을 포함한 구성입니다.

「오늘의 준비」 셀은 Prophet 을 7번 적합해 30초 안팎 걸리고 아래 셋을 출력합니다.

- 표 1 · 예측 원점 5개의 168시간 RMSE (대): 원점마다 재적합해 168시간을 한 번에 예측한 값
- 표 2 · 일 수요 365일 추세의 변화량이 큰 변화점 수 (개): 연간 계절성을 포함한 적합과 제외한 적합의 개수. 첫 행의 잠재적 변화점은 Prophet 이 자료의 처음 80% 구간에 고르게 깔아 두는 후보 25곳입니다. 그 가운데 추세 성장률 변화량의 절댓값이 0.01 보다 큰 곳만 변화량이 큰 변화점으로 집계합니다. 0.01 은 수요와 시간을 0～1 로 줄여 적합한 단위의 값입니다
- 그림 1 · 일 수요와 두 적합의 추세 (대/일)

예측 원점 5개는 10월 19일～11월 16일의 금요일 23시입니다. **재적합(refit)** 은 원점마다 9월 1일부터 그 원점까지의 실측으로 계수를 다시 추정하는 것입니다.

In [ ]:
# 오늘의 준비 — 읽고 실행만 합니다
# 서울 공유자전거로 표 1 · 표 2 · 그림 1 을 만듭니다. Prophet 을 7번 적합해 30초 안팎 걸립니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
hol_days = sorted(set(df.index[df["holiday"].eq("Holiday")].normalize()))
holidays_one = pd.DataFrame({"holiday": "공휴일", "ds": hol_days})   # 공휴일 18일을 이름 하나로 묶은 표

# 표 1: 금요일 23시 예측 원점 5개마다 9월 1일부터 원점까지로 재적합해 168시간을 한 번에 예측한다
origins = pd.date_range("2018-10-19 23:00", "2018-11-16 23:00", freq="7D")
origin_rmse = []                          # (원점, 계절 나이브 168, Prophet 기본 구성)
for o in origins:
    fit_part = df.loc["2018-09-01 00:00":o, "demand"]
    week = pd.date_range(o + pd.Timedelta(hours=1), periods=168, freq="h")
    m = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True,
                holidays=holidays_one, uncertainty_samples=0)          # 구간 없이 예측값만 낸다
    m.fit(pd.DataFrame({"ds": fit_part.index, "y": fit_part.values}))
    y_week = df.loc[week, "demand"]
    origin_rmse.append((o, rmse(y_week, df["demand"].shift(168).loc[week]),    # 1주일 전 같은 시각 값
                        rmse(y_week, m.predict(pd.DataFrame({"ds": week}))["yhat"])))
sn5_mean = float(np.mean([r[1] for r in origin_rmse]))
p5_mean = float(np.mean([r[2] for r in origin_rmse]))
print("표 1 · 예측 원점 5개의 168시간 RMSE (대)")
print("  예측 원점        계절 나이브 168   Prophet 기본 구성")
for o, sn, pr in origin_rmse:
    print(f"  {o.month:>2}월 {o.day:>2}일 23시   {sn:15.2f}   {pr:17.2f}")
print(f"  평균             {sn5_mean:15.2f}   {p5_mean:17.2f}")

# 표 2 · 그림 1: 일 수요 365일에 연간 계절성을 포함한 적합과 제외한 적합, 잠재적 변화점의 성장률 변화량을 센다
_sum = df["demand"].resample("D").sum()                     # 보정 수요의 하루 합계
daily = pd.DataFrame({"ds": _sum.index, "y": _sum.values})
cp_count, daily_trend = {}, {}
for yearly in (True, False):
    m = Prophet(yearly_seasonality=yearly, weekly_seasonality=True, daily_seasonality=False,
                uncertainty_samples=0)                      # 일 수요에는 하루 안의 주기가 없다
    m.fit(daily)
    delta = m.params["delta"].mean(axis=0)                  # 잠재적 변화점마다 성장률 변화량
    cp_count[yearly] = int((np.abs(delta) > 0.01).sum())
    daily_trend[yearly] = m.predict(daily)["trend"].values
print("표 2 · 일 수요 365일 추세의 변화량이 큰 변화점 수 (개)")
print(f"  잠재적 변화점          {len(m.changepoints)}")
print(f"  연간 계절성 포함 적합  {cp_count[True]}")
print(f"  연간 계절성 제외 적합  {cp_count[False]}")

fig, ax = plt.subplots(figsize=(FIG_W, 3.4))
ax.plot(daily["ds"], daily["y"], color=COL["연회색"], lw=0.8, label="일 수요")
ax.plot(daily["ds"], daily_trend[True], color=COL["남색"], lw=2.0, label="추세 (연간 계절성 포함 적합)")
ax.plot(daily["ds"], daily_trend[False], color=COL["하늘색"], lw=2.0, label="추세 (연간 계절성 제외 적합)")
ax.set_title("그림 1 · 일 수요와 두 적합의 추세 (대/일)")
ax.set_ylabel("하루 합계 (대/일)")
datefmt(ax, "%Y-%m", 7)
ax.legend(loc="upper left")
plt.show()

## 예측구간과 외생 변수 확인

> **[문제] 80% 구간이 한쪽으로만 벗어났다면 무엇부터 고치나**
>
> 예를 들어 80% 예측구간이 검증 100시각 가운데 60시각만 담았고, 벗어난 40시각이 모두 구간 아래였다고 합시다.
> 
> **핵심 키워드:** 벗어난 방향, 예측선의 치우침, 명목 포함률과 실제 포함률을 함께 보고
>
> 1. `interval_width` 를 0.95 로 높여 구간을 넓힌다
> 2. 예측선이 실측보다 높게 치우쳤으니 모형 구성부터 고친다
> 3. 실측을 절반 넘게 담았으니 구간을 그대로 보고한다

> **[문제] 금요일 23시에 다음 168시간을 예측할 때 더할 수 있는 열은**
>
> 예를 들어 11월 16일(금) 23시에 다음 168시간의 수요를 예측하며, 아래 열 가운데 하나를 `add_regressor` 로 더하려 합니다. 학습 구간에서는 세 열 모두 수요와 상관이 높았다고 합시다.
> 
> **핵심 키워드:** 예측 원점에 채울 수 있는 값, 상관의 크기는 기준이 아니다, 사후 예측
>
> 1. 시간별 기온: 기상청 예보로 다음 7일 값을 채울 수 있다
> 2. 대여소 반납 건수: 수요와의 상관이 세 열 가운데 가장 높다
> 3. 공유자전거 앱 접속자 수: 수요보다 한두 시간 먼저 움직인다

## 서울 공유자전거 평가표로 판단하기

아래 두 문항은 「오늘의 준비」 셀 출력을 읽고 답합니다. 표 1 은 예측 원점 5개의 채점표입니다. 표 2 와 그림 1 은 일 수요의 추세를 두 구성으로 적합한 결과입니다. 그림 1 에서 추세선이 꺾인 곳은 변화량이 큰 변화점, 곧 추세 기울기가 바뀐 곳입니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_013.png" alt="분할마다 학습 구간과 검증 구간을 가로 막대로 그린 그림" width="560" height="280"/>
<p align="center"><em>▲ 시계열 교차 검증(time series cross-validation). 분할 0～3 에서 파란 막대가 학습, 붉은 막대가 검증이고 검증은 언제나 학습 뒤에 온다. 표 1 의 예측 원점 5개도 이 구조로 채점했다. 맨 아래 class·group 줄은 보지 않는다 (출처: scikit-learn 공식 문서, Visualizing cross-validation behavior)</em></p>

<img src="https://facebook.github.io/prophet/static/trend_changepoints_files/trend_changepoints_9_0.png" alt="관측 점과 예측 곡선에 추세선과 세로 점선 12개를 겹쳐 그린 그림" width="560" height="333"/>
<p align="center"><em>▲ Prophet 공식 문서의 위키백과 페이지뷰(로그) 계열. 검은 점이 관측, 빨간 실선이 추세, 빨간 세로 점선 12개가 변화량이 큰 변화점이다. 변화점은 달력의 사건이 아니라 추세의 기울기가 바뀐 시점이다 (출처: Prophet 공식 문서, Trend Changepoints)</em></p>

> **[문제] 원점 5개 평가표로 내리는 투입 판단**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 1 · 예측 원점 5개의 168시간 RMSE (대)」를 봅니다. Prophet 기본 구성을 운영 예측에 투입할까요?
> 
> **핵심 키워드:** 원점 5개 평균이 투입 기준, 한 주의 순위는 원점마다 바뀐다, 운영 예측은 베이스라인
>
> 1. 투입한다. 11월 9일 원점 한 주에서 베이스라인보다 크게 덜 빗나갔다
> 2. 판단할 수 없다. 원점마다 순위가 바뀌니 평균도 믿을 수 없다
> 3. 투입하지 않는다. 원점 5개 평균 RMSE 가 베이스라인보다 크다

> **[문제] 연간 계절성을 제외하자 생긴 꺾임을 보고할 때**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 2 · 일 수요 365일 추세의 변화량이 큰 변화점 수 (개)」와 「그림 1 · 일 수요와 두 적합의 추세 (대/일)」를 함께 봅니다. 꺾임은 추세 기울기가 바뀐 곳입니다. 운영팀장에게 보고할 때 맞는 읽기는?
> 
> **핵심 키워드:** 같은 데이터 다른 성분 구성, 변화점은 사건이 아니다, 개수에는 구성을 함께 적는다
>
> 1. 연간 계절성을 제외한 적합의 꺾임이 진짜 추세다. 계절 항에 숨어 있던 꺾임이 드러났다
> 2. 꺾임은 모형에 포함한 성분에 따라 달라지니 개수와 함께 성분 구성을 적는다
> 3. 두 적합의 개수가 다르니 둘 다 믿을 수 없어 변화점은 보고에서 뺀다

## 미션 1 — 80% 예측구간의 실제 포함률

표 1 로 보면 Prophet 기본 구성은 아직 투입하지 않습니다. 그래도 Prophet 의 예측구간을 참고로 보고하려면, 그 구간이 실측을 얼마나 담았는지부터 확인합니다.

#### 할 일 — 미션 1 (25분)

1. 아래 「미션 1 준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 「PROMPT 셀」의 PROMPT 에 시킬 일을 내 말로 적습니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. LMS 밖이면 받은 코드를 AI 코드 표시선 아래에 둡니다.
4. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: Prophet 기본 구성의 80% 예측구간은 검증 168시간의 실측을 80% 만큼 담았을까요?

<img src="https://otexts.com/fpp3/fpp_files/figure-html/prophetfc-1.png" alt="반시간 단위 전력 수요의 실측 뒤로 2일 예측과 두 겹의 예측구간을 그린 그림" width="600" height="370"/>
<p align="center"><em>▲ 반시간 단위 전력 수요의 2일 예측. 진한 파란 영역이 80% 예측구간, 연한 영역이 95% 예측구간이다. 명목 포함률을 높이면 구간이 길어진다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3판, 그림 12.12)</em></p>

**쓸 수 있는 것**: 「미션 1 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `train` · `valid` | 학습 1,848시간(9월 1일～11월 16일)과 검증 168시간(11월 17～23일)의 시간 표. 열 `demand` 가 보정 수요(대) |
| `holidays_one` | 휴일 데이터프레임. 공휴일 18일을 이름 하나로 묶은 표(열 `holiday`·`ds`) |
| `SEED` | 시드. 표본 경로는 Prophet 이 추세 변화와 잡음을 무작위로 1,000번 뽑아 만든 미래 궤적이다. 80% 예측구간은 그 궤적들의 10%·90% 지점이라, 예측 직전에 시드를 고정해야 실행마다 같은 구간이 나온다. 적합은 난수를 쓰지 않으니 적합 직전 고정은 완성본과 차례를 맞추는 것이다 |

#### 만들 것

Prophet 기본 구성을 `train` 으로 적합하고 `valid` 168시각을 예측합니다. 굵은 이름은 채점이 찾는 이름이라 글자 그대로 씁니다.

1. **명목 포함률**: 단위 % · 정수. 코드에 숫자로 적지 않고 적합한 모형의 `interval_width` 에서 읽습니다
2. **실제 포함률**: 단위 % · 소수 둘째 자리. 실측이 `yhat_lower` 이상 `yhat_upper` 이하인 시각의 비율입니다. 168시간 가운데 몇 시간인지도 같은 행에 적습니다
3. **대조용 한 행**: 구간 아래로 벗어난 시각 수와 위로 벗어난 시각 수. 채점하지 않고 완성본과 비교할 때 씁니다
4. **그래프 한 장**: 가로축 「검증 168시간」, 세로축 「수요(대)」. 실측선·예측선 2개와 옅게 칠한 80% 예측구간

**통과 기준**: 1·2번이 「이름 = 값 %」로 한 행씩 출력되고 4번 그래프가 한 장 있으면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름**: 위 표의 이름과 설치 셀의 `rmse(a, b)` 는 다시 만들지 말라고 적습니다.
- **구성과 시드**: 성분 구성과, 적합 직전·예측 직전에 `np.random.seed(SEED)` 를 부르라는 것을 적습니다.
- **명목 포함률**: 95 나 80 을 손으로 적지 말고 모형에서 읽으라고 적습니다.

<details class="lms-extra"><summary>자세히 — 예시 프롬프트</summary>

위 표의 이름과 rmse 는 새로 만들지 말고, train 의 demand 로 Prophet 을 연간 계절성은 제외하고 주간과 일간 계절성, holidays_one 휴일은 포함해 적합해 valid 168시각을 예측해 줘. 적합 직전과 예측 직전에는 np.random.seed(SEED) 를 불러 줘. "명목 포함률 = 값%" 는 모형의 interval_width 로, "실제 포함률 = 값% (몇/168시간)" 은 실측이 구간 안인 비율로 한 행씩 출력하고 구간 아래·위로 벗어난 시각 수도 찍어 줘. 실측선·예측선과 예측구간을 그린 그래프 한 장에 가로축·세로축 이름을 달아 줘.

</details>

#### 예상 적기

> **[예측] 이 80% 예측구간은 검증 168시간 가운데 몇 % 를 담을까요?**
>
> 1. 80% 안팎이다. 명목 포함률만큼 담는다
> 2. 80% 보다 많이 담는다. 이 주의 오차가 학습 구간의 오차보다 작다
> 3. 80% 보다 적게 담는다. 예측선이 한쪽으로 치우친다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로

train = df.loc["2018-09-01 00:00":"2018-11-16 23:00"]      # 학습 1,848시간
valid = df.loc["2018-11-17 00:00":"2018-11-23 23:00"]      # 검증 168시간
hol_days = sorted(set(df.index[df["holiday"].eq("Holiday")].normalize()))
holidays_one = pd.DataFrame({"holiday": "공휴일", "ds": hol_days})   # 공휴일 18일을 이름 하나로 묶은 표

print(f"학습 {len(train):,}시간  {train.index[0]:%m-%d %H시} ~ {train.index[-1]:%m-%d %H시}")
print(f"검증 {len(valid):,}시간  {valid.index[0]:%m-%d %H시} ~ {valid.index[-1]:%m-%d %H시}")
print(f"휴일 데이터프레임 {len(holidays_one)}행, 열 {list(holidays_one.columns)}")
print(f"시드 SEED = {SEED}: 예측구간은 표본 경로로 만들어 적합 직전과 예측 직전에 한 번씩 고정합니다")
print("쓸 수 있는 이름 : train · valid (시간 표, 열 demand) · holidays_one (휴일 데이터프레임)"
      " · SEED (시드) · rmse(a, b)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 미션 2 — 기온·강수를 더한 검증 RMSE

미션 1 의 성분 구성에 기온·강수를 외생 변수로 더하면 오차가 줄어드는지 봅니다. 미션 1 과 같은 순서로 합니다.

**이 미션의 질문**: 기온·강수를 더하면 검증 168시간 RMSE 가 낮아지고, 운영에서 두 열의 미래 값을 채울 수 있을까요?

**쓸 수 있는 것**: 「미션 2 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `train` · `valid` · `holidays_one` | 미션 1 과 같은 표. 열 `temp` 는 기온(도), `rain` 은 강수(mm)의 실측 |
| `REGS` | 더할 두 열의 이름 `rain`·`temp` |
| `BASE_RMSE` · `SNAIVE_RMSE` | 날씨 없는 Prophet 기본 구성과 계절 나이브 168 의 검증 168시간 RMSE(대) |

#### 만들 것

1. **날씨를 더한 검증 168시간 RMSE**: 단위 대 · 소수 둘째 자리. `REGS` 두 열을 `add_regressor` 로 더해 다시 적합합니다. 학습 표와 예측 표 양쪽에 두 열을 두고, 예측 표에는 검증 구간의 실측 날씨를 대입합니다
2. **기온 계수**: 단위 대/도 · 소수 둘째 자리. 기온이 1도 오를 때 수요가 몇 대 바뀌는지를 원래 단위(대/도)로 읽은 값입니다. Prophet 은 기온·강수를 평균 0·표준편차 1 로, 수요를 가장 큰 값으로 나눠 바꾼 뒤 적합합니다. 그래서 `m.params` 의 계수는 바뀐 단위의 값이고, `regressor_coefficients(m)` 가 이를 기온 1도당 대수로 복원해 반환합니다
3. **판단 한 행**: 이번 검증 구간에 날씨 열을 더할지와, 운영에서 두 열의 미래 값을 어디서 채울지. 채점하지 않습니다
4. **그래프 한 장**: 가로 막대 3개와 가로축 「검증 168시간 RMSE(대)」. 막대 이름은 「계절 나이브」·「날씨 없음」·「날씨 있음」입니다

**통과 기준**: 1·2번이 「이름 = 값 단위」로 한 행씩 출력되고 4번 그래프가 있으면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **바꾸지 말 것**: 미션 1 의 성분 구성과 시드 고정은 그대로 두고 `REGS` 두 열만 더하라고 적습니다.
- **두 표**: 학습 표와 예측 표 모두에 `rain`·`temp` 열을 두라고 적습니다.
- **계수 읽기**: `m.params` 의 계수를 손으로 복원하지 말고 `from prophet.utilities import regressor_coefficients` 로 불러 읽으라고 적습니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/elecdemand2-1.png" alt="기온을 가로축, 전력 수요를 세로축에 둔 산점도로 가운데가 낮고 양쪽이 높은 모양" width="560" height="345"/>
<p align="center"><em>▲ 전력 수요를 기온에 흩뿌린 그림. 15～20도 근처가 가장 낮고 양쪽으로 커진다. 기온에 계수 하나를 곱하는 직선 항은 이런 관계를 담지 못한다. 점의 색(근무일 여부)은 여기서 보지 않는다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3판, 그림 12.6)</em></p>

#### 예상 적기

「미션 2 준비 셀」이 찍는 학습 구간과 검증 구간의 기온을 먼저 봅니다.

> **[예측] 기온·강수를 더하면 검증 168시간 RMSE 는 `BASE_RMSE` 와 비교해 어떻게 될까요?**
>
> 1. 낮아진다. 날씨는 수요와 함께 움직인다
> 2. 거의 같다. 두 열의 기여분이 작다
> 3. 커진다. 기온 항이 예측을 지나치게 옮긴다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로

train = df.loc["2018-09-01 00:00":"2018-11-16 23:00"]      # 학습 1,848시간
valid = df.loc["2018-11-17 00:00":"2018-11-23 23:00"]      # 검증 168시간
hol_days = sorted(set(df.index[df["holiday"].eq("Holiday")].normalize()))
holidays_one = pd.DataFrame({"holiday": "공휴일", "ds": hol_days})   # 공휴일 18일을 이름 하나로 묶은 표
REGS = ["rain", "temp"]                                    # 더할 외생 변수 두 열

#  비교할 두 RMSE 를 이 셀에서 다시 만든다. 날씨 없는 Prophet 기본 구성과 1주일 전 같은 시각 값
m_base = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True,
                 holidays=holidays_one, uncertainty_samples=0)
m_base.fit(pd.DataFrame({"ds": train.index, "y": train["demand"].values}))
BASE_RMSE = rmse(valid["demand"], m_base.predict(pd.DataFrame({"ds": valid.index}))["yhat"])
SNAIVE_RMSE = rmse(valid["demand"], df["demand"].shift(168).loc[valid.index])

q05 = train["temp"].quantile(0.05)                        # 학습 기온을 낮은 쪽부터 정렬한 5% 지점
print(f"학습 {len(train):,}시간 · 검증 {len(valid)}시간 · 휴일 데이터프레임 {len(holidays_one)}행")
print(f"기온 평균: 학습 {train['temp'].mean():.2f}도, 검증 {valid['temp'].mean():.2f}도")
print(f"검증 168시간 가운데 학습 기온의 5% 지점({q05:.1f}도)보다 추운 시각: "
      f"{int((valid['temp'] < q05).sum())}시간")
print(f"비 온 시각: 학습 {int((train['rain'] > 0).sum())}시간, 검증 {int((valid['rain'] > 0).sum())}시간")
print(f"비교할 두 RMSE: 날씨 없는 Prophet 기본 구성 {BASE_RMSE:.2f}대, 계절 나이브 168 {SNAIVE_RMSE:.2f}대")
print("쓸 수 있는 이름 : train · valid (시간 표, 열 demand·temp·rain) · holidays_one · REGS (더할 두 열)"
      " · BASE_RMSE · SNAIVE_RMSE (비교할 두 RMSE, 대) · SEED (시드) · rmse(a, b)")

In [ ]:
# 미션 2 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 오늘의 결론

<div class="lms-summary">

**정리**

- 예측 원점 5개 평균 RMSE: Prophet 기본 구성 약 381대, 계절 나이브 168 363.55대. 11월 9일 한 주에서는 Prophet 이 작았지만 평균은 베이스라인이 작습니다
- 일 수요 추세의 변화량이 큰 변화점은 연간 계절성을 포함하면 0곳, 제외하면 7곳이라 개수에는 모형에 포함한 성분을 함께 적습니다
- 80% 예측구간은 11월 17～23일 168시간의 약 90% 를 담았고, 벗어난 시각은 위아래 양쪽에 있었습니다
- 기온·강수를 더하면 실측 날씨를 대입한 사후 예측에서도 검증 RMSE 가 약 310대에서 약 340대로 커졌습니다
- 할 일: 운영 예측은 `df["demand"].shift(168)` 로 냅니다. Prophet 구간을 함께 보고할 때는 `m.interval_width` 로 읽은 명목 포함률과 실제 포함률을 함께 적고, 날씨 열은 이번 검증 구간에 더하지 않습니다

Prophet 은 아직 투입하지 않고 운영 예측은 베이스라인(계절 나이브 168)으로 둡니다.

</div>

> **[과제] 실습 제출 — 80% 예측구간의 실제 포함률과 날씨 외생 변수**
>
> 미션 1·2 의 「PROMPT 셀」이 채점을 통과한 상태로 노트북을 제출합니다. PROMPT 는 내가 쓴 그대로 둡니다.
>
> 마감: 2026-10-14T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 미션 1 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
> | 미션 2 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 시계열 다섯째 날 완료</h2>
<div style="color:#656565">예측 원점 5개로 Prophet 의 투입 여부를 정하고, 80% 예측구간의 실제 포함률과 기온·강수 외생 변수의 효과를 직접 확인했습니다.</div>
</div>